<a href="https://colab.research.google.com/github/Kommmi/Chain_GQS/blob/main/spectrum_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eigenstates and eigenvalues of the chain: comparing models

Exact diagonalization of the fixed-$N$ sector for two models:

* **coulomb**: the original notebooks, $V_{ij} = V/|i-j|$, no site energies.
* **townsend2021**: Townsend et al. (arXiv:2011.07113): $V_{ij} = \lambda_{ee}(1 - f_{ex}\delta_{|i-j|,1})/(|i-j| + \zeta)$ plus the core potential, with $\lambda_{nuc} = \lambda_{ee}$.

Hopping $t = J = 1$, so energies are in units of the hopping and the interaction strength is $V/t$ or $\lambda_{ee}/t$. All functions come from `chain_gqs`; change parameters in the next cell only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chain_gqs import coulomb, townsend2021, exact_spectrum, spectrum_sweep, projected_ensemble

## Parameters

In [ ]:
N = 2
L_values = [4, 5]
strength = 1.0                          # interaction strength for the single-point comparisons
sweep_values = np.logspace(-2, 3, 61)   # interaction strengths for the sweeps (units of t)

# name -> (model factory taking L, name of its interaction parameter)
models = {
    "coulomb":      (lambda L: coulomb(L=L, N=N, J=1.0, V=strength), "V"),
    "townsend2021": (lambda L: townsend2021(L=L, N=N, lam_ee=strength, t=1.0), "lam_ee"),
}

## 1. One model in detail

`Spectrum.table()` lists the energies, the excitation energies $E - E_0$ and degeneracies.
`vectors_table()` gives the eigenvector amplitudes on each occupation configuration (site order $|n_0 n_1 \dots\rangle$);
each column is fixed so its largest amplitude is positive. `densities_table()` gives $\langle n_i\rangle$ for each eigenstate.

In [ ]:
model_name, L = "townsend2021", 4
model = models[model_name][0](L)
spec = exact_spectrum(model)
print(model.label)
spec.table()

In [ ]:
spec.vectors_table()

In [ ]:
spec.densities_table()

## 2. Both models at the same interaction strength

In [ ]:
rows = {}
for L in L_values:
    for name, (make, _) in models.items():
        rows[(L, name)] = exact_spectrum(make(L)).energies
comparison = pd.DataFrame({k: pd.Series(v) for k, v in rows.items()})
comparison.columns = pd.MultiIndex.from_tuples(comparison.columns, names=["L", "model"])
comparison.index.name = "level"
comparison.round(4).fillna("")   # blank: level does not exist for that L

In [ ]:
# Ground-state site occupations <n_i> for each model
gs_density = {}
for L in L_values:
    for name, (make, _) in models.items():
        gs_density[(L, name)] = exact_spectrum(make(L)).site_densities()[0]
pd.DataFrame({k: pd.Series(v) for k, v in gs_density.items()}).T.rename(columns=lambda i: f"<n{i}>").round(3).fillna("")

## 3. Spectrum versus interaction strength

Excitation energies $E_k - E_0$ on log–log axes, as in Fig. 1 of the paper. Weak interaction is on the left, strong on the right.
In the coulomb model only $V$ changes; in townsend2021 the core potential grows with $\lambda_{ee}$ as well.

In [ ]:
sweeps = {(L, name): spectrum_sweep(make(L), param, sweep_values)
          for L in L_values for name, (make, param) in models.items()}

fig, axes = plt.subplots(len(L_values), len(models), figsize=(11, 4 * len(L_values)),
                         sharex=True, layout="constrained")
for i, L in enumerate(L_values):
    for j, name in enumerate(models):
        ax, sw = axes[i, j], sweeps[(L, name)]
        exc = sw["energies"][:, 1:] - sw["energies"][:, :1]
        ax.loglog(sw["values"], np.clip(exc, 1e-6, None), color="tab:blue", lw=1)
        ax.set(title=f"{name}, L={L}, N={N}", xlabel=f"{sw['param']} / t", ylabel=r"$E_k - E_0$ (units of t)")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for (L, name), sw in sweeps.items():
    ax.loglog(sw["values"], np.clip(sw["gap"], 1e-8, None),
              ls="-" if name == "townsend2021" else "--", label=f"{name}, L={L}")
ax.set(xlabel="interaction strength / t", ylabel=r"gap $E_1 - E_0$", title="Lowest excitation energy")
ax.legend()
plt.show()

## 4. How the ground state changes with interaction

Ground-state weight on each occupation configuration, $|\langle c|\Psi_0\rangle|^2$, at a few interaction strengths.

The chain is mirror symmetric, so $\langle n_i\rangle = \langle n_{L-1-i}\rangle$ and only one site of each mirror pair is plotted. Dashed: coulomb; solid: townsend2021.

In [ ]:
snapshot_values = [0.01, 1.0, 10.0, 100.0]
for name, (make, param) in models.items():
    for L in L_values:
        base = make(L)
        weights = {v: np.abs(exact_spectrum(base.with_(**{param: v})).vectors[:, 0]) ** 2
                   for v in snapshot_values}
        df = pd.DataFrame(weights, index=pd.Index(base.sector_labels, name="configuration"))
        df.columns = [f"{param}/t={v:g}" for v in snapshot_values]
        print(f"{name}, L={L}")
        display(df.round(3))

In [ ]:
fig, axes = plt.subplots(1, len(L_values), figsize=(12, 4), layout="constrained")
for ax, L in zip(axes, L_values):
    for name in models:
        sw = sweeps[(L, name)]
        dens = np.array([s.site_densities()[0] for s in sw["spectra"]])
        for i in range((L + 1) // 2):     # mirror symmetry: <n_i> = <n_(L-1-i)>
            pair = f"<n{i}>" if i == L - 1 - i else f"<n{i}> = <n{L - 1 - i}>"
            ax.semilogx(sw["values"], dens[:, i], ls="-" if name == "townsend2021" else "--",
                        color=f"C{i}", label=f"{pair} {name}")
    ax.set(title=f"Ground-state occupations, L={L}", xlabel="interaction strength / t", ylabel=r"$\langle n_i\rangle$")
    ax.legend(fontsize=7)
plt.show()

## 5. Projected ensemble of an eigenstate

An eigenstate does not evolve (apart from a global phase), so its projected ensemble is fixed in time.
This gives the stationary reference point for the dynamics studied elsewhere.

In [ ]:
model_name, L, level = "townsend2021", 5, 0
model = models[model_name][0](L)
psi = exact_spectrum(model).full_vector(level, L)
ens = projected_ensemble(psi, L, subsystem=(0, 1), N=N)
display(ens.sector_table(style="cm"))
display(ens.ensemble_table(style="cm"))

## 6. Checks

In [ ]:
for L in L_values:
    for name, (make, _) in models.items():
        m = make(L)
        s = exact_spectrum(m)
        H = m.sector_hamiltonian
        residual = max(np.linalg.norm(H @ s.vectors[:, k] - s.energies[k] * s.vectors[:, k])
                       for k in range(len(s.energies)))
        orth = np.abs(s.vectors.T @ s.vectors - np.eye(len(s.energies))).max()
        print(f"{name:13s} L={L}: max |Hv - Ev| = {residual:.1e}, max orthonormality error = {orth:.1e}")